## Explicação dos gradientes encontrados no código

(i) O que está implementado (`backward.cu#L258`):

```cpp
float denom = a * c - b * b;
dL_da = denom2inv * (-c * c * dL_dconic.x + 2 * b * c * dL_dconic.y + (denom - a * c) * dL_dconic.z);
dL_dc = denom2inv * (-a * a * dL_dconic.z + 2 * a * b * dL_dconic.y + (denom - a * c) * dL_dconic.x);
dL_db = denom2inv * 2 * (b * c * dL_dconic.x - (denom + 2 * b * b) * dL_dconic.y + a * b * dL_dconic.z);
```

Onde $\Sigma' = \begin{bmatrix}a & b\\b&c\end{bmatrix}$ conhecer $ \frac{\partial \mathcal{L}}{\partial \Sigma'}$ significa conhecer $\frac{\partial \mathcal{L}}{\partial a}, \frac{\partial \mathcal{L}}{\partial b}, \frac{\partial \mathcal{L}}{\partial c}$.

E ainda, conic = $\Sigma'^{-1}= \begin{bmatrix}x & y\\y&z\end{bmatrix}$:


$$
\frac{\partial \mathcal{L}}{\partial a} =  
\frac{1}{(ac-b^2)^2} (\frac{\partial \mathcal{L}}{\partial x} + 2bc\frac{\partial \mathcal{L}}{\partial y} + -b^2\frac{\partial \mathcal{L}}{\partial z} )
$$
$$
\frac{\partial \mathcal{L}}{\partial a} = 
\underbrace{\begin{bmatrix} 
\frac{-c^2}{(ac-b^2)^2} & \frac{bc}{(ac-b^2)^2} & \frac{bc}{(ac-b^2)^2} & \frac{-b^2}{(ac-b^2)^2}
\end{bmatrix}}_{\frac{\partial \Sigma'^{-1}}{\partial a}}
\underbrace{
\begin{bmatrix} 
\partial \mathcal{L}/\partial x \\ \partial \mathcal{L}/\partial y \\ \partial \mathcal{L}/\partial y \\ \partial \mathcal{L}/\partial z
\end{bmatrix}}_{\frac{\partial \mathcal{L}}{\partial \Sigma'^{-1}}}
$$



Por sua vez dL/dconic vem de: 

```cpp
// Update gradients w.r.t. 2D covariance (2x2 matrix, symmetric)
atomicAdd(&dL_dconic2D[global_id].x, -0.5f * gdx * d.x * dL_dG);
atomicAdd(&dL_dconic2D[global_id].y, -0.5f * gdx * d.y * dL_dG);
atomicAdd(&dL_dconic2D[global_id].w, -0.5f * gdy * d.y * dL_dG);
```

$$
\partial \mathcal{L}/\partial x = -0,5 \cdot G \cdot d_x \cdot d_x \cdot \partial \mathcal{L}/\partial G
$$
$$
\partial \mathcal{L}/\partial y = -0,5 \cdot G \cdot d_x \cdot d_y \cdot \partial \mathcal{L}/\partial G
$$
$$
\partial \mathcal{L}/\partial z = -0,5 \cdot G \cdot d_y \cdot d_y \cdot \partial \mathcal{L}/\partial G
$$


Traduzindo:

$$
\frac{\partial \mathcal{L}}{\partial a} = 
\begin{bmatrix} 
\frac{-c^2}{(ac-b^2)^2} & \frac{bc}{(ac-b^2)^2} & \frac{bc}{(ac-b^2)^2} & \frac{-b^2}{(ac-b^2)^2}
\end{bmatrix}
\begin{bmatrix} 
-0,5  G  d_x^2 \cdot \partial \mathcal{L}/\partial G \\ 
-0,5  G  d_x d_y \cdot \partial \mathcal{L}/\partial G \\ 
-0,5  G  d_x d_y \cdot \partial \mathcal{L}/\partial G \\ 
-0,5  G  d_y^2 \cdot \partial \mathcal{L}/\partial G
\end{bmatrix}
$$
$$
\frac{\partial \mathcal{L}}{\partial a} = 
\begin{bmatrix} 
\frac{-c^2}{(ac-b^2)^2} & \frac{bc}{(ac-b^2)^2} & \frac{bc}{(ac-b^2)^2} & \frac{-b^2}{(ac-b^2)^2}
\end{bmatrix}
\begin{bmatrix} 
-0,5  G  d_x^2  \\ 
-0,5  G  d_x d_y  \\ 
-0,5  G  d_x d_y  \\ 
-0,5  G  d_y^2 
\end{bmatrix} \frac{\partial \mathcal{L}}{\partial G}
$$
$$
\frac{\partial \mathcal{L}}{\partial a} = 
\frac{\partial \Sigma'^{-1}}{\partial a}
\frac{\partial \mathcal{L}}{\partial \Sigma'^{-1}} =
\frac{\partial \Sigma'^{-1}}{\partial a}
\frac{\partial G}{\partial \Sigma'^{-1}}\frac{\partial \mathcal{L}}{\partial G}
$$
$$
\frac{\partial \mathcal{L}}{\partial b} = 
\frac{\partial \Sigma'^{-1}}{\partial b}
\frac{\partial G}{\partial \Sigma'^{-1}}\frac{\partial \mathcal{L}}{\partial G}
$$
$$
\frac{\partial \mathcal{L}}{\partial c} = 
\frac{\partial \Sigma'^{-1}}{\partial c}
\frac{\partial G}{\partial \Sigma'^{-1}}\frac{\partial \mathcal{L}}{\partial G}
$$


Por sua vez dL/dG vem de (`backward.cu#L619`): 

```cpp
const float dL_dG = opacity * dL_dalpha;
```

Por sua vez dL/dalpha vem de (`backward.cu#L579`): 
```cpp
// Propagate gradients to per-Gaussian colors and keep
// gradients w.r.t. alpha (blending factor for a Gaussian/pixel
// pair).
float dL_dalpha = 0.0f;
const int global_id = collected_id[j];
for (int ch = 0; ch < C; ch++)
{
    const float c = collected_colors[ch * BLOCK_SIZE + j];
    // Update last color (to be used in the next iteration)
    accum_rec[ch] = last_alpha * last_color[ch] + (1.f - last_alpha) * accum_rec[ch];
    last_color[ch] = c;
    const float dL_dchannel = dL_dpixel[ch];
    dL_dalpha += (c - accum_rec[ch]) * dL_dchannel;
...
```

Ou seja: ............ 


## Análise de derivada pela regra da cadeia e diretamente

(i) dG/dcov2D sem vetorização:

$$\frac{\partial G}{\partial \Sigma'} = \frac{\partial G}{\partial \Sigma'^{-1}}  \cdot \frac{\partial \Sigma'^{-1}}{\partial \Sigma'}$$

Sendo:

$$\Sigma' = 
\begin{bmatrix}
a & b \\
b & c
\end{bmatrix},
\Sigma'^{-1} = 
\begin{bmatrix}
x & y \\
y & z
\end{bmatrix} = 
\begin{bmatrix}
\frac{c}{a c - b^2} & \frac{-b}{a c - b^2} \\
\frac{-b}{a c - b^2} & \frac{a}{a c - b^2}
\end{bmatrix},
G = exp(-0,5(d_x^2 x + 2d_x d_y y + d_y^2z))$$

$$
\frac{\partial \Sigma'^{-1}}{\partial \Sigma'} = 
\begin{bmatrix}
\frac{\partial \Sigma'^{-1}}{\partial a} & \frac{\partial \Sigma'^{-1}}{\partial b} \\ \\
\frac{\partial \Sigma'^{-1}}{\partial b} & \frac{\partial z}{\partial c}
\end{bmatrix} = 
\begin{bmatrix}
    \begin{bmatrix}
        \frac{\partial x}{\partial a} & \frac{\partial y}{\partial a} \\
        \frac{\partial y}{\partial a} & \frac{\partial z}{\partial a}
    \end{bmatrix} & \begin{bmatrix}
                        \frac{\partial x}{\partial b} & \frac{\partial y}{\partial b} \\
                        \frac{\partial y}{\partial b} & \frac{\partial z}{\partial b}
                    \end{bmatrix} \\
    \begin{bmatrix}
        \frac{\partial x}{\partial b} & \frac{\partial y}{\partial b} \\
        \frac{\partial y}{\partial b} & \frac{\partial z}{\partial b}
    \end{bmatrix} & 
    \begin{bmatrix}
        \frac{\partial x}{\partial c} & \frac{\partial y}{\partial c} \\
        \frac{\partial y}{\partial c} & \frac{\partial z}{\partial c}
    \end{bmatrix}  \\
\end{bmatrix} \in \mathbb{R}^{2\times 2 \times 2 \times 2}
$$

$$
\frac{\partial \Sigma'^{-1}}{\partial a} = 
\begin{bmatrix}
\frac{\partial}{\partial a}(\frac{c}{ac-b^2}) & \frac{\partial}{\partial a}(\frac{-b}{ac-b^2}) \\
\frac{\partial}{\partial a}(\frac{-b}{ac-b^2}) & \frac{\partial}{\partial a}(\frac{a}{ac-b^2})
\end{bmatrix} = 
\begin{bmatrix}
- \frac{c^2}{(ac-b^2)^2} & \frac{bc}{(ac-b^2)^2} \\
\frac{bc}{(ac-b^2)^2} & - \frac{b^2}{(ac-b^2)^2}
\end{bmatrix}
$$

$$
\frac{\partial \Sigma'^{-1}}{\partial b} = 
\begin{bmatrix}
- \frac{2bc}{(ac-b^2)^2} & \frac{-ac-b^2}{(ac-b^2)^2} \\
\frac{-ac-b^2}{(ac-b^2)^2} & \frac{2ab}{(ac-b^2)^2}
\end{bmatrix}
$$

$$
\frac{\partial \Sigma'^{-1}}{\partial c} = 
\begin{bmatrix}
- \frac{b^2}{(ac-b^2)^2} & \frac{ab}{(ac-b^2)^2} \\
\frac{ab}{(ac-b^2)^2} & - \frac{a^2}{(ac-b^2)^2}
\end{bmatrix}
$$

(ii) dG/dcov2D com vetorização

$$
\begin{bmatrix}
\frac{\partial G}{\partial a} & \frac{\partial G}{\partial b} \\ \\
\frac{\partial G}{\partial b} & \frac{\partial G}{\partial c}
\end{bmatrix}_{2\times 2} = 
\begin{bmatrix}
\frac{\partial G}{\partial x} & \frac{\partial G}{\partial y} \\ \\
\frac{\partial G}{\partial y} & \frac{\partial G}{\partial z}
\end{bmatrix}_{2\times 2} 
\begin{bmatrix}
\left[\frac{\partial \Sigma'^{-1}}{\partial a}\right] & \left[\frac{\partial \Sigma'^{-1}}{\partial b}\right] \\ \\
\left[\frac{\partial \Sigma'^{-1}}{\partial b}\right] & \left[\frac{\partial \Sigma'^{-1}}{\partial c}\right]
\end{bmatrix}_{2 \times 2 \times 2 \times 2} 
$$

Redesenhando as matrizes:

$$
\begin{bmatrix}
\frac{\partial G}{\partial a} & \frac{\partial G}{\partial b} &
\frac{\partial G}{\partial b} & \frac{\partial G}{\partial c}
\end{bmatrix}_{1\times 4} = 
\begin{bmatrix}
\frac{\partial G}{\partial x} & \frac{\partial G}{\partial y} &
\frac{\partial G}{\partial y} & \frac{\partial G}{\partial z}
\end{bmatrix}_{1\times 4} 
\begin{bmatrix}
\left[\frac{\partial \Sigma'^{-1}}{\partial a}\right] & \left[\frac{\partial \Sigma'^{-1}}{\partial b}\right] &
\left[\frac{\partial \Sigma'^{-1}}{\partial b}\right] & \left[\frac{\partial \Sigma'^{-1}}{\partial c}\right]
\end{bmatrix}_{4 \times 4} 
$$

$$
\begin{bmatrix}
\frac{\partial G}{\partial a} & \frac{\partial G}{\partial b} & \frac{\partial G}{\partial c}
\end{bmatrix}_{1\times 3} = 
\begin{bmatrix}
\frac{\partial G}{\partial x} & \frac{\partial G}{\partial y} & \frac{\partial G}{\partial z}
\end{bmatrix}_{1\times 3} 
\begin{bmatrix}
\begin{bmatrix}
\frac{\partial x}{\partial a}\\
\frac{\partial y}{\partial a}\\
\frac{\partial z}{\partial a} 
\end{bmatrix} & 
\begin{bmatrix}
\frac{\partial x}{\partial b}\\
\frac{\partial y}{\partial b}\\
\frac{\partial z}{\partial b} 
\end{bmatrix} & 
\begin{bmatrix}
\frac{\partial x}{\partial c}\\
\frac{\partial y}{\partial c}\\
\frac{\partial z}{\partial c} 
\end{bmatrix}
\end{bmatrix}_{3 \times 3} 
$$

Assim:

$$
\begin{bmatrix}
\frac{\partial G}{\partial a} & \frac{\partial G}{\partial b} & \frac{\partial G}{\partial c}
\end{bmatrix} = 
$$
$$
-G \begin{bmatrix}
0,5d_x^2 & d_x d_y & 0,5d_y^2
\end{bmatrix}
\begin{bmatrix}
    \begin{bmatrix}
        - \frac{c^2}{(ac-b^2)^2} \\ 
        \frac{bc}{(ac-b^2)^2} \\
        -\frac{b^2}{(ac-b^2)^2}
    \end{bmatrix} & 
    \begin{bmatrix}
        - \frac{2bc}{(ac-b^2)^2} \\ 
        \frac{-ac-b^2}{(ac-b^2)^2} \\
        \frac{2ab}{(ac-b^2)^2}
    \end{bmatrix} &
    \begin{bmatrix}
        - \frac{b^2}{(ac-b^2)^2} \\ 
        \frac{ab}{(ac-b^2)^2} \\
        -\frac{a^2}{(ac-b^2)^2}
    \end{bmatrix}
\end{bmatrix}
$$

Focando apenas em dG/da (o resultado deve ser um escalar)

$$\frac{\partial G}{\partial a} = 
-G 
\begin{bmatrix}
    0,5d_x^2 & d_x d_y & 0,5d_y^2
\end{bmatrix}
\begin{bmatrix}
    - \frac{c^2}{(ac-b^2)^2} \\ 
    \frac{bc}{(ac-b^2)^2} \\ 
    -\frac{b^2}{(ac-b^2)^2}
\end{bmatrix}
$$

$$\frac{\partial G}{\partial a} = 
-G \left [0,5d_x^2 \left (- \frac{c^2}{(ac-b^2)^2} \right ) + d_x d_y \frac{bc}{(ac-b^2)^2} + 0,5d_y^2 \left (-\frac{b^2}{(ac-b^2)^2} \right ) \right ]
$$

$$\frac{\partial G}{\partial a} = 
G \left [0,5d_x^2 \frac{c^2}{(ac-b^2)^2} - d_x d_y \frac{bc}{(ac-b^2)^2} + 0,5d_y^2 \frac{b^2}{(ac-b^2)^2}  \right ]
$$

$$\frac{\partial G}{\partial a} = 
G (0,5 )\left [\frac{c^2}{(ac-b^2)^2}d_x^2  - \frac{bc}{(ac-b^2)^2}2d_x d_y  +  \frac{b^2}{(ac-b^2)^2}d_y^2  \right ]
$$

Diretamente:
$$
G = exp(-0,5(d_x^2 (\frac{c}{a c - b^2}) + 2d_x d_y (\frac{-b}{a c - b^2}) + d_y^2(\frac{a}{a c - b^2})))
$$
$$
\frac{\partial G}{\partial a} = G (0,5) \left (\frac{c^2}{(ac-b^2)^2}d_x^2 - \frac{bc}{(ac-b^2)^2}2d_xd_y +\frac{b^2}{(ac-b^2)^2}d_y^2 \right )
$$

## Cálculo do gradiente da função densidade de probabilidade gaussiana 2D em relação à inversa da matriz de covariância 2D $\left (\frac{\partial G}{\partial \Sigma'^{-1}} \right )$:

### Dados:

#### (i) $\Sigma = RSS^TR^T = RS^2R^T$

Onde: 

- $R \in \mathbb{R}^{3 \times 3} $ é ortonormal;

- $S \in \mathbb{R}^{3 \times 3}$ é diagonal; e

- $S = \begin{bmatrix}s_x & 0 & 0 \\ 0 & s_y & 0 \\ 0 & 0 & s_z\end{bmatrix} \rightarrow SS^T= S^2$

#### (ii) $\Sigma' = JW\Sigma W^TJ^T$

Onde: 

- $J = \begin{bmatrix}f_x/Z_C & 0 & -f_xX_C/Z^2_C \\0 & f_y/Z_C & -f_yY_C/Z^2_C \\ 0 & 0 & 0 \end{bmatrix} $; e

- $W \in \mathbb{R}^{3 \times 3}$ é ortonormal ;

Pode-se abreviar seus termos:

$\Sigma' = \begin{bmatrix}a & b & 0\\ b& c& 0\\ 0 & 0 & 0\end{bmatrix} \rightarrow \Sigma' = \begin{bmatrix}a & b\\ b& c\end{bmatrix}$

Sua inversa fica:

$\Sigma'^{-1} = 
\begin{bmatrix}
x & y \\
y & z
\end{bmatrix} = 
\begin{bmatrix}
\frac{c}{a c - b^2} & \frac{-b}{a c - b^2} \\
\frac{-b}{a c - b^2} & \frac{a}{a c - b^2}
\end{bmatrix}$

#### (iii) $\sigma_{\mu} = \frac{\min\{ s^2_x, s^2_y, s^2_z\}}{s^2_x + s^2_y + s^2_z}$

#### (iv) $G = \sigma_{\mu} exp(-0,5(d_x^2 x + 2d_x d_y y + d_y^2z))$

Onde: 

- $d_x$ e $d_y$ são constantes.

### Desenvolvimento:

#### (i) Derivada por elemento de $\Sigma'^{-1}$

$$
\frac{\partial G}{\partial \Sigma'^{-1}} = 
\begin{bmatrix}
\frac{\partial G}{\partial x} & \frac{\partial G}{\partial y} \\
\frac{\partial G}{\partial y} & \frac{\partial G}{\partial z}
\end{bmatrix}
$$

Aplicando a regra do produto: $\frac{\partial [f(w)g(w)]}{\partial w} = g(w)\frac{\partial f(w)}{\partial w} + f(w)\frac{\partial g(w)}{\partial w} $

$
\frac{\partial G}{\partial x} = \frac{\partial [\sigma_{\mu} exp(-0,5(d_x^2 x + 2d_x d_y y + d_y^2z))]}{\partial x} \rightarrow
$

- $
\frac{\partial G}{\partial x} = \underbrace{\frac{\partial \sigma_{\mu}}{\partial x}}_{(\text{ii.1})} \frac{G}{\sigma_{\mu}} + \cancel{\sigma_{\mu}} \frac{G}{ \cancel{\sigma_{\mu}}} (-0,5d_x^2)
$

- $
\frac{\partial G}{\partial y} = \underbrace{\frac{\partial \sigma_{\mu}}{\partial y}}_{(\text{ii.2})} \frac{G}{\sigma_{\mu}} + \cancel{\sigma_{\mu}} \frac{G}{ \cancel{\sigma_{\mu}}} (- d_x d_y)
$

- $
\frac{\partial G}{\partial z} = \underbrace{\frac{\partial \sigma_{\mu}}{\partial z}}_{(\text{ii.3})} \frac{G}{\sigma_{\mu}} + \cancel{\sigma_{\mu}} \frac{G}{ \cancel{\sigma_{\mu}}} (-0,5d_y^2)
$

#### (ii.1) $\frac{\partial \sigma_{\mu}}{\partial x}$

$$
\frac{\partial \sigma_{\mu}}{\partial x}= \underbrace{\frac{\partial \sigma_{\mu}}{\partial s}}_{\text{iii.1}} \underbrace{\frac{\partial s}{\partial x}}_{\text{iv.1}} = \left [ \frac{\partial \sigma_{\mu}}{\partial s_{x}} \frac{\partial \sigma_{\mu}}{\partial s_{y}} \frac{\partial \sigma_{\mu}}{\partial s_{z}} \right ] 
\begin{bmatrix} 
\frac{\partial s_x}{\partial x}  \\
\frac{\partial s_y}{\partial x} \\
\frac{\partial s_z}{\partial x}
\end{bmatrix} = 
\frac{\partial \sigma_{\mu}}{\partial s_{x}}\frac{\partial s_x}{\partial x} + 
\frac{\partial \sigma_{\mu}}{\partial s_{y}}\frac{\partial s_y}{\partial x} +
\frac{\partial \sigma_{\mu}}{\partial s_{z}}\frac{\partial s_z}{\partial x}
$$

$$
\frac{\partial \sigma_{\mu}}{\partial x}= \frac{\partial \sigma_{\mu}}{\partial s} \frac{\partial s}{\partial x} = \left [ \frac{\partial \sigma_{\mu}}{\partial s_{x}} \frac{\partial \sigma_{\mu}}{\partial s_{y}} \frac{\partial \sigma_{\mu}}{\partial s_{z}} \right ] 
\begin{bmatrix} 
\frac{\partial s_x}{\partial x}  \\
\frac{\partial s_y}{\partial x} \\
\frac{\partial s_z}{\partial x}
\end{bmatrix} = 
\frac{\partial \sigma_{\mu}}{\partial s_{x}}\frac{\partial s_x}{\partial x} + 
\frac{\partial \sigma_{\mu}}{\partial s_{y}}\frac{\partial s_y}{\partial x} +
\frac{\partial \sigma_{\mu}}{\partial s_{z}}\frac{\partial s_z}{\partial x}
$$

#### (iii.1) $\frac{\partial \sigma_{\mu}}{\partial s}$

$
\frac{\partial \sigma_{\mu}}{\partial s} = \frac{\partial \left [ \frac{\min\{ s^2_x, s^2_y, s^2_z\}}{s^2_x + s^2_y + s^2_z} \right ]}{\partial s} \rightarrow
$ min{} não é diferenciável em qualquer ponto, portanto precisamos testar os casos:

In [ ]:
import sympy as sp

: 

$
\frac{\partial \sigma_{\mu}}{\partial s} = \left [ \frac{\partial \sigma_{\mu}}{\partial s_x} \frac{\partial \sigma_{\mu}}{\partial s_y} \frac{\partial \sigma_{\mu}}{\partial s_z} \right ] \rightarrow
$ 

- Se $s^2_x$ é o min:

$
\frac{\partial \sigma_{\mu}}{\partial s} = 
\left [ 
    \frac{2s_x(s^2_y+s^2_z)}{s_x^2+s_y^2+s_z^2}  \ \ 
    \frac{2s_y(s^2_x)}{s_x^2+s_y^2+s_z^2}  \ \ 
    \frac{2s_z(s^2_x)}{s_x^2+s_y^2+s_z^2} 
\right ]
$ 

- Se $s^2_y$ é o min:

$
\frac{\partial \sigma_{\mu}}{\partial s} = 
\left [ 
    \frac{2s_x(s^2_y)}{s_x^2+s_y^2+s_z^2}  \ \ 
    \frac{2s_y(s^2_x+s^2_z)}{s_x^2+s_y^2+s_z^2}  \ \ 
    \frac{2s_z(s^2_y)}{s_x^2+s_y^2+s_z^2} 
\right ]
$ 

- Se $s^2_z$ é o min:

$
\frac{\partial \sigma_{\mu}}{\partial s} = 
\left [ 
    \frac{2s_x(s^2_z)}{s_x^2+s_y^2+s_z^2}  \ \ 
    \frac{2s_y(s^2_z)}{s_x^2+s_y^2+s_z^2}  \ \ 
    \frac{2s_z(s^2_x+s^2_y)}{s_x^2+s_y^2+s_z^2} 
\right ]
$ 

#### (iv.1) $\frac{\partial s}{\partial x}$

Aplicando a regra da derivada inversa: $\frac{\partial [f(w)]}{\partial w} =\left (\frac{\partial w}{\partial f(w)}\right )^{-1}$ 

$
\frac{\partial s}{\partial x} = \left (\frac{\partial x}{\partial s} \right )^{-1}
$

Por sua vez, aplicando a regra da cadeia em $\frac{\partial x}{\partial s}$:

$
\frac{\partial x}{\partial s} = 
\frac{\partial x}{\partial \Sigma}
\frac{\partial \Sigma}{\partial s} =
 \underbrace{\frac{\partial x}{\partial \Sigma'}}_{\text{iv.1.a}}  \ \ 
 \underbrace{\frac{\partial  \Sigma'}{\partial \Sigma}}_{\text{iv.1.b}} \ \ 
  \underbrace{\frac{\partial \Sigma}{\partial s}}_{\text{iv.1.c}}
$

$ \displaystyle
\frac{\partial s}{\partial x} = 
\left(\frac{\partial x}{\partial s}\right)^{-1} = 
\left(\frac{\partial x}{\partial \Sigma}
\frac{\partial \Sigma}{\partial s}\right)^{-1} =
\left(\frac{\partial x}{\partial \Sigma'} \ \ 
\frac{\partial  \Sigma'}{\partial \Sigma}\ \ 
\frac{\partial \Sigma}{\partial s}\right)^{-1}
$

Contraindo $\Sigma \in \mathbb{R}^{3 \times 3}$ para um vetor: $vec(\Sigma) \in \mathbb{R}^9$:

$
\frac{\partial x}{\partial s} = 
\frac{\partial x}{\partial vec(\Sigma)} \cdot
\frac{\partial vec(\Sigma)}{\partial s}=
 \left [\frac{\partial x}{\partial vec(\Sigma)} \right ]_{1 \times 9}
 \begin{bmatrix} 
 \frac{\partial vec(\Sigma)}{\partial s_{x}} \ \ 
 \frac{\partial vec(\Sigma)}{\partial s_{y}} \ \ 
 \frac{\partial vec(\Sigma)}{\partial s_{z}} 
 \end{bmatrix}_{9 \times 3}
$

Por sua vez: 

$\left [\frac{\partial x}{\partial vec(\Sigma)} \right ]_{1 \times 9} = 
\left [\frac{\partial x}{\partial vec(\Sigma')} \right ]_{1 \times 9}  \ \ 
\left [\frac{\partial vec(\Sigma')}{\partial vec(\Sigma)} \right ]_{9 \times 9} \ \ $

#### (iv.1.a) $\frac{\partial x}{\partial \Sigma'}$

Sabendo que $x= \frac{c}{a c - b^2}$ aplica-se a derivada por elemento de $\Sigma'$:

$$
\frac{\partial x}{\partial \Sigma'}= 
\begin{bmatrix}
\frac{\partial x}{\partial a} & \frac{\partial x}{\partial b} & 0 \\ \\
\frac{\partial x}{\partial b} & \frac{\partial x}{\partial c} & 0 \\ \\
0 & 0 & 0 
\end{bmatrix} = 
\begin{bmatrix}
- \frac{c^2}{(ac-b^2)^2} &  \frac{2bc}{(ac-b^2)^2} & 0 \\
\frac{2bc}{(ac-b^2)^2} & - \frac{b^2}{(ac-b^2)^2} & 0 \\ \\
0 & 0 & 0 
\end{bmatrix}
$$

$\left [\frac{\partial x}{\partial vec(\Sigma')} \right ]_{1 \times 9} = 
\begin{bmatrix}
-\frac{c^2}{(ac-b^2)^2} &  \frac{2bc}{(ac-b^2)^2} & 0 &\frac{2bc}{(ac-b^2)^2} & - \frac{b^2}{(ac-b^2)^2} & 0 & 0 & 0 & 0
\end{bmatrix}$

In [38]:
import sympy as sp
Cov2D = sp.Matrix([['a', 'b'], ['b', 'c']])
Cov2D_inv = Cov2D.inv()
Cov2D_inv_expanded = sp.Matrix(3,3, lambda i,j: Cov2D_inv[i,j] if j<2 and i<2 else 0)
x = Cov2D_inv_expanded[0,0]
y = Cov2D_inv_expanded[0,1]
z = Cov2D_inv_expanded[1,1]
dx = sp.diff(x, Cov2D)
dx_dCov2D = sp.Matrix([[dx[0,0], dx[0,1], 0, dx[1,0], sp.factor(dx[1,1]), 0, 0, 0, 0]]) # necessário usar sp.factor pra forçar a simplificação

# Adiantando a derivada dy_dCov2D (iv.2.a)
dy = sp.diff(y, Cov2D)
dy_dCov2D = sp.Matrix([[dy[0,0], sp.factor(dy[0,1]), 0, sp.factor(dy[1,0]), dy[1,1], 0, 0, 0, 0]])

#Adiantando a derivada dz_dCov2D (iv.3.a)
dz = sp.diff(z, Cov2D)
dz_dCov2D = sp.Matrix([[sp.factor(dz[0,0]), dz[0,1], 0, dz[1,0], dz[1,1], 0, 0, 0, 0]])

dx_dCov2D

Matrix([[-c**2/(a*c - b**2)**2, 2*b*c/(a*c - b**2)**2, 0, 2*b*c/(a*c - b**2)**2, -b**2/(a*c - b**2)**2, 0, 0, 0, 0]])

#### (iv.1.b) $\frac{\partial \Sigma'}{\partial \Sigma}$

$$\frac{\partial  \Sigma'}{\partial \Sigma} = \frac{\partial [JW\Sigma W^TJ^T]}{\partial \Sigma} \rightarrow \frac{\partial  vec(\Sigma')}{\partial vec(\Sigma)} =  JW \otimes JW$$


$\left [\frac{\partial vec(\Sigma')}{\partial vec(\Sigma)} \right ]_{9 \times 9}=
\begin{bmatrix}
\frac{\partial a}{\partial vec(\Sigma)} \\ \frac{\partial b}{\partial vec(\Sigma)} \\ 0 \\
\frac{\partial b}{\partial vec(\Sigma)} \\ \frac{\partial c}{\partial vec(\Sigma)} \\ 0 \\
0 \\ 0 \\ 0 
\end{bmatrix}$

$\left [\frac{\partial vec(\Sigma')}{\partial vec(\Sigma)} \right ]_{9 \times 9}=$

In [42]:
import sympy as sp

# Recriando as matrizes explicitamente com símbolos elementares para J, W e S
J = sp.Matrix(3, 3, lambda i, j: 0 if (i==0 and j==1) or (i==1 and j==0) or (i==2 and j==0) or (i==2 and j==1) or (i==2 and j==2) else sp.Symbol(f'J{i+1}{j+1}'))

# J = sp.Matrix(3, 3, lambda i, j: sp.Symbol(f'J{i+1}{j+1}'))
W = sp.Matrix(3, 3, lambda i, j: sp.Symbol(f'W{i+1}{j+1}'))
S = sp.Matrix(3, 3, lambda i, j: sp.Symbol(f'S{i+1}{j+1}'))

# Expressão para X
X = J * W * S * W.T * J.T

# Expande a multiplicação
X_explicit = sp.expand(X)
# print(sp.latex(X_explicit))
# print()
# X_explicit
d = sp.diff(X, S)
factored_expression = sp.factor(d)

# print(sp.latex(factored_expression))
print()
factored_expression

[[[[(J11*W11 + J13*W31)**2, (J11*W11 + J13*W31)*(J22*W21 + J23*W31), 0], [(J11*W11 + J13*W31)*(J22*W21 + J23*W31), (J22*W21 + J23*W31)**2, 0], [0, 0, 0]], [[(J11*W11 + J13*W31)*(J11*W12 + J13*W32), (J11*W11 + J13*W31)*(J22*W22 + J23*W32), 0], [(J11*W12 + J13*W32)*(J22*W21 + J23*W31), (J22*W21 + J23*W31)*(J22*W22 + J23*W32), 0], [0, 0, 0]], [[(J11*W11 + J13*W31)*(J11*W13 + J13*W33), (J11*W11 + J13*W31)*(J22*W23 + J23*W33), 0], [(J11*W13 + J13*W33)*(J22*W21 + J23*W31), (J22*W21 + J23*W31)*(J22*W23 + J23*W33), 0], [0, 0, 0]]], [[[(J11*W11 + J13*W31)*(J11*W12 + J13*W32), (J11*W12 + J13*W32)*(J22*W21 + J23*W31), 0], [(J11*W11 + J13*W31)*(J22*W22 + J23*W32), (J22*W21 + J23*W31)*(J22*W22 + J23*W32), 0], [0, 0, 0]], [[(J11*W12 + J13*W32)**2, (J11*W12 + J13*W32)*(J22*W22 + J23*W32), 0], [(J11*W12 + J13*W32)*(J22*W22 + J23*W32), (J22*W22 + J23*W32)**2, 0], [0, 0, 0]], [[(J11*W12 + J13*W32)*(J11*W13 + J13*W33), (J11*W12 + J13*W32)*(J22*W23 + J23*W33), 0], [(J11*W13 + J13*W33)*(J22*W22 + J23*W32),

Ou ainda melhor:

$\left [\frac{\partial vec(\Sigma')}{\partial vec(\Sigma)} \right ]_{9 \times 9}=$

In [44]:
# Define A = JW
A = J * W

# Define X = A * S * A.T
X = A * S * A.T
# Vetoriza X e S
vec_X = sp.Matrix(X).reshape(9, 1)
vec_S = sp.Matrix(S).reshape(9, 1)

# Derivada Jacobiana: d(vec_X)/d(vec_S)^T = A ⊗ A
dCov2D_dCov3D = sp.kronecker_product(A, A)
# print(sp.latex(dCov2D_dCov3D))
print()
dCov2D_dCov3D

Matrix([
[                 (J11*W11 + J13*W31)**2, (J11*W11 + J13*W31)*(J11*W12 + J13*W32), (J11*W11 + J13*W31)*(J11*W13 + J13*W33), (J11*W11 + J13*W31)*(J11*W12 + J13*W32),                  (J11*W12 + J13*W32)**2, (J11*W12 + J13*W32)*(J11*W13 + J13*W33), (J11*W11 + J13*W31)*(J11*W13 + J13*W33), (J11*W12 + J13*W32)*(J11*W13 + J13*W33),                  (J11*W13 + J13*W33)**2],
[(J11*W11 + J13*W31)*(J22*W21 + J23*W31), (J11*W11 + J13*W31)*(J22*W22 + J23*W32), (J11*W11 + J13*W31)*(J22*W23 + J23*W33), (J11*W12 + J13*W32)*(J22*W21 + J23*W31), (J11*W12 + J13*W32)*(J22*W22 + J23*W32), (J11*W12 + J13*W32)*(J22*W23 + J23*W33), (J11*W13 + J13*W33)*(J22*W21 + J23*W31), (J11*W13 + J13*W33)*(J22*W22 + J23*W32), (J11*W13 + J13*W33)*(J22*W23 + J23*W33)],
[                                      0,                                       0,                                       0,                                       0,                                       0,                                       0,   

#### (iv.1.c) $\frac{\partial \Sigma}{\partial s}$

$\frac{\partial \Sigma}{\partial s} = \frac{\partial [RS^2R^T]}{\partial s} \rightarrow$

$
\frac{\partial \Sigma}{\partial s} = R \left [ \frac{\partial  S^2}{\partial s_x} \frac{\partial S^2}{\partial s_y} \frac{\partial S^2}{\partial s_z} \right ] R^T =
R \left [ 
    \begin{bmatrix} 
    2s_x & 0 & 0 \\
    0 & 0 & 0 \\
    0 & 0 & 0
    \end{bmatrix} \ \
    \begin{bmatrix} 
    0 & 0 & 0 \\
    0 & 2s_y & 0 \\
    0 & 0 & 0
    \end{bmatrix} \ \
    \begin{bmatrix} 
    0 & 0 & 0 \\
    0 & 0 & 0 \\
    0 & 0 & 2s_z
    \end{bmatrix}
    \right ] 
R^T
$

$\frac{\partial \Sigma}{\partial s_x} = 
\begin{bmatrix}
R_{11} & R_{12} & R_{13} \\
R_{21} & R_{22} & R_{23} \\
R_{31} & R_{32} & R_{33}
\end{bmatrix}
\begin{bmatrix}
2s_x & 0 & 0 \\
0 & 0 & 0 \\
0 & 0 & 0
\end{bmatrix}
\begin{bmatrix}
R_{11} & R_{21} & R_{31} \\
R_{12} & R_{22} & R_{32} \\
R_{13} & R_{23} & R_{33}
\end{bmatrix}=2s_x
\begin{bmatrix}
R_{11} & 0 & 0 \\
R_{21} & 0 & 0 \\
R_{31} & 0 & 0
\end{bmatrix}\begin{bmatrix}
R_{11} & R_{21} & R_{31} \\
R_{12} & R_{22} & R_{32} \\
R_{13} & R_{23} & R_{33}
\end{bmatrix}
$

$\frac{\partial \Sigma}{\partial s_x} = 
2s_x
\begin{bmatrix}
R_{11}\\
R_{21}\\
R_{31}
\end{bmatrix}\begin{bmatrix}
R_{11} & R_{21} & R_{31} 
\end{bmatrix}=2s_x
\bf{r_1}\bf{r_1^T}=2s_x
\begin{bmatrix}
R_{11}^2 & R_{11}R_{21} & R_{11}R_{31} \\
R_{21}R_{11} & R_{21}^2 & R_{21}R_{31} \\
R_{31}R_{11} & R_{31}R_{21} & R_{31}^2
\end{bmatrix}
$

Analogamente:

$\frac{\partial \Sigma}{\partial s_y} = 
2s_y
\bf{r_2}\bf{r_2^T}=2s_y
\begin{bmatrix}
R_{12}\\
R_{22}\\
R_{32}
\end{bmatrix}\begin{bmatrix}
R_{12} & R_{22} & R_{32} 
\end{bmatrix} =2s_y
\begin{bmatrix}
R_{12}^2 & R_{12}R_{22} & R_{12}R_{32} \\
R_{22}R_{12} & R_{22}^2 & R_{22}R_{32} \\
R_{32}R_{12} & R_{32}R_{22} & R_{32}^2
\end{bmatrix}
$

$\frac{\partial \Sigma}{\partial s_z} = 
2s_z
\bf{r_3}\bf{r_3^T}=2s_z
\begin{bmatrix}
R_{13}\\
R_{23}\\
R_{33}
\end{bmatrix}\begin{bmatrix}
R_{13} & R_{23} & R_{33} 
\end{bmatrix} =2s_z
\begin{bmatrix}
R_{13}^2 & R_{13}R_{23} & R_{13}R_{33} \\
R_{23}R_{13} & R_{23}^2 & R_{23}R_{33} \\
R_{33}R_{13} & R_{33}R_{23} & R_{33}^2
\end{bmatrix}
$

Assim: 

$\frac{\partial \Sigma}{\partial s}=\left [
2s_x\begin{bmatrix}
R_{11}^2 & R_{11}R_{21} & R_{11}R_{31} \\
R_{21}R_{11} & R_{21}^2 & R_{21}R_{31} \\
R_{31}R_{11} & R_{31}R_{21} & R_{31}^2
\end{bmatrix}, \ 
2s_y\begin{bmatrix}
R_{12}^2 & R_{12}R_{22} & R_{12}R_{32} \\
R_{22}R_{12} & R_{22}^2 & R_{22}R_{32} \\
R_{32}R_{12} & R_{32}R_{22} & R_{32}^2
\end{bmatrix}, \ 
2s_z\begin{bmatrix}
R_{13}^2 & R_{13}R_{23} & R_{13}R_{33} \\
R_{23}R_{13} & R_{23}^2 & R_{23}R_{33} \\
R_{33}R_{13} & R_{33}R_{23} & R_{33}^2
\end{bmatrix}
\right ]^T
$

Checagem:

$\frac{\partial \Sigma}{\partial s} = $

In [48]:
s = ['s_x','s_y','s_z']
S = sp.Matrix(3, 3, lambda i, j: 0 if (i!=j) else sp.Symbol(s[i]))
R = sp.Matrix(3, 3, lambda i, j: sp.Symbol(f'R{i+1}{j+1}'))
s = sp.Matrix(3, 1, lambda i, j: sp.Symbol(s[i]))
Cov3D = R*S*S.T*R.T
dCov3D_ds = sp.diff(Cov3D, s)
dCov3D_ds = dCov3D_ds.reshape(9,3)
# print(sp.latex(dCov3D_ds))
print()
dCov3D_ds

[[2*R11**2*s_x, 2*R11*R21*s_x, 2*R11*R31*s_x], [2*R11*R21*s_x, 2*R21**2*s_x, 2*R21*R31*s_x], [2*R11*R31*s_x, 2*R21*R31*s_x, 2*R31**2*s_x], [2*R12**2*s_y, 2*R12*R22*s_y, 2*R12*R32*s_y], [2*R12*R22*s_y, 2*R22**2*s_y, 2*R22*R32*s_y], [2*R12*R32*s_y, 2*R22*R32*s_y, 2*R32**2*s_y], [2*R13**2*s_z, 2*R13*R23*s_z, 2*R13*R33*s_z], [2*R13*R23*s_z, 2*R23**2*s_z, 2*R23*R33*s_z], [2*R13*R33*s_z, 2*R23*R33*s_z, 2*R33**2*s_z]]

#### (v.1) Resultado de $\frac{\partial x}{\partial s} $

$
\underbrace{\frac{\partial x}{\partial s}}_{1 \times 3} = 
 \underbrace{\frac{\partial x}{\partial \Sigma'}}_{1 \times 9}  \ \ 
 \underbrace{\frac{\partial  \Sigma'}{\partial \Sigma}}_{9 \times 9} \ \ 
 \underbrace{\frac{\partial \Sigma}{\partial s}}_{9 \times 3} =
$

In [51]:
print(dx_dCov2D.shape, dCov2D_dCov3D.shape, dCov3D_ds.shape)
print(type(dx_dCov2D), type(dCov2D_dCov3D), type(dCov3D_ds)) # Por algum motivo não são do mesmo tipo e não dá pra multiplicá-las
print(type(dx_dCov2D), type(sp.Matrix(dCov2D_dCov3D)), type(sp.Matrix(dCov3D_ds)))
print()
dx_ds = dx_dCov2D * sp.Matrix(dCov2D_dCov3D) * sp.Matrix(dCov3D_ds)
# print(sp.latex(dx_ds))
print()
dx_ds.T

(1, 9) (9, 9) (9, 3)
<class 'sympy.matrices.dense.MutableDenseMatrix'> <class 'sympy.matrices.immutable.ImmutableDenseMatrix'> <class 'sympy.tensor.array.dense_ndim_array.MutableDenseNDimArray'>
<class 'sympy.matrices.dense.MutableDenseMatrix'> <class 'sympy.matrices.dense.MutableDenseMatrix'> <class 'sympy.matrices.dense.MutableDenseMatrix'>




Matrix([
[2*R11**2*s_x*(-b**2*(J22*W21 + J23*W31)**2/(a*c - b**2)**2 + 4*b*c*(J11*W11 + J13*W31)*(J22*W21 + J23*W31)/(a*c - b**2)**2 - c**2*(J11*W11 + J13*W31)**2/(a*c - b**2)**2) + 2*R11*R21*s_x*(-b**2*(J22*W21 + J23*W31)*(J22*W22 + J23*W32)/(a*c - b**2)**2 + 2*b*c*(J11*W11 + J13*W31)*(J22*W22 + J23*W32)/(a*c - b**2)**2 + 2*b*c*(J11*W12 + J13*W32)*(J22*W21 + J23*W31)/(a*c - b**2)**2 - c**2*(J11*W11 + J13*W31)*(J11*W12 + J13*W32)/(a*c - b**2)**2) + 2*R11*R31*s_x*(-b**2*(J22*W21 + J23*W31)*(J22*W23 + J23*W33)/(a*c - b**2)**2 + 2*b*c*(J11*W11 + J13*W31)*(J22*W23 + J23*W33)/(a*c - b**2)**2 + 2*b*c*(J11*W13 + J13*W33)*(J22*W21 + J23*W31)/(a*c - b**2)**2 - c**2*(J11*W11 + J13*W31)*(J11*W13 + J13*W33)/(a*c - b**2)**2) + 2*R12**2*s_y*(-b**2*(J22*W21 + J23*W31)*(J22*W22 + J23*W32)/(a*c - b**2)**2 + 2*b*c*(J11*W11 + J13*W31)*(J22*W22 + J23*W32)/(a*c - b**2)**2 + 2*b*c*(J11*W12 + J13*W32)*(J22*W21 + J23*W31)/(a*c - b**2)**2 - c**2*(J11*W11 + J13*W31)*(J11*W12 + J13*W32)/(a*c - b**2)**2) + 2*R12*

#### (vi.1) Resultado de $\frac{\partial s}{\partial x} $

In [53]:
ds_dx = sp.Matrix([1/dx_ds[0,0], 1/dx_ds[0,1], 1/dx_ds[0,2]])
print(dx_ds.shape, ds_dx.shape)
# print(sp.latex(ds_dx))
print()
ds_dx

(1, 3) (3, 1)



Matrix([
[1/(2*R11**2*s_x*(-b**2*(J22*W21 + J23*W31)**2/(a*c - b**2)**2 + 4*b*c*(J11*W11 + J13*W31)*(J22*W21 + J23*W31)/(a*c - b**2)**2 - c**2*(J11*W11 + J13*W31)**2/(a*c - b**2)**2) + 2*R11*R21*s_x*(-b**2*(J22*W21 + J23*W31)*(J22*W22 + J23*W32)/(a*c - b**2)**2 + 2*b*c*(J11*W11 + J13*W31)*(J22*W22 + J23*W32)/(a*c - b**2)**2 + 2*b*c*(J11*W12 + J13*W32)*(J22*W21 + J23*W31)/(a*c - b**2)**2 - c**2*(J11*W11 + J13*W31)*(J11*W12 + J13*W32)/(a*c - b**2)**2) + 2*R11*R31*s_x*(-b**2*(J22*W21 + J23*W31)*(J22*W23 + J23*W33)/(a*c - b**2)**2 + 2*b*c*(J11*W11 + J13*W31)*(J22*W23 + J23*W33)/(a*c - b**2)**2 + 2*b*c*(J11*W13 + J13*W33)*(J22*W21 + J23*W31)/(a*c - b**2)**2 - c**2*(J11*W11 + J13*W31)*(J11*W13 + J13*W33)/(a*c - b**2)**2) + 2*R12**2*s_y*(-b**2*(J22*W21 + J23*W31)*(J22*W22 + J23*W32)/(a*c - b**2)**2 + 2*b*c*(J11*W11 + J13*W31)*(J22*W22 + J23*W32)/(a*c - b**2)**2 + 2*b*c*(J11*W12 + J13*W32)*(J22*W21 + J23*W31)/(a*c - b**2)**2 - c**2*(J11*W11 + J13*W31)*(J11*W12 + J13*W32)/(a*c - b**2)**2) + 2*R

#### (ii.2) $\frac{\partial \sigma_{\mu}}{\partial y}$ analogamente a (ii.1)

#### (iii.2) = (iii.1) $\frac{\partial \sigma_{\mu}}{\partial s}$

#### (iv.2) $\frac{\partial s}{\partial y}$ analogamente a (iv.1)

#### (iv.2.a) $\frac{\partial y}{\partial \Sigma'}$ analogamente a (iv.1.a)

#### (iv.2.b) = (iv.1.b) $\frac{\partial  \Sigma'}{\partial  \Sigma}$

#### (iv.2.c) = (iv.1.c) $\frac{\partial  \Sigma}{\partial s}$

#### (v.2) Resultado de $\frac{\partial y}{\partial s} $

In [55]:
dy_ds = dy_dCov2D * sp.Matrix(dCov2D_dCov3D) * sp.Matrix(dCov3D_ds)
# print(sp.latex(dy_ds))
print()
dy_ds.T

Matrix([
[2*R11**2*s_x*(a*b*(J22*W21 + J23*W31)**2/(a*c - b**2)**2 + b*c*(J11*W11 + J13*W31)**2/(a*c - b**2)**2 - 2*(J11*W11 + J13*W31)*(J22*W21 + J23*W31)*(a*c + b**2)/(a*c - b**2)**2) + 2*R11*R21*s_x*(a*b*(J22*W21 + J23*W31)*(J22*W22 + J23*W32)/(a*c - b**2)**2 + b*c*(J11*W11 + J13*W31)*(J11*W12 + J13*W32)/(a*c - b**2)**2 - (J11*W11 + J13*W31)*(J22*W22 + J23*W32)*(a*c + b**2)/(a*c - b**2)**2 - (J11*W12 + J13*W32)*(J22*W21 + J23*W31)*(a*c + b**2)/(a*c - b**2)**2) + 2*R11*R31*s_x*(a*b*(J22*W21 + J23*W31)*(J22*W23 + J23*W33)/(a*c - b**2)**2 + b*c*(J11*W11 + J13*W31)*(J11*W13 + J13*W33)/(a*c - b**2)**2 - (J11*W11 + J13*W31)*(J22*W23 + J23*W33)*(a*c + b**2)/(a*c - b**2)**2 - (J11*W13 + J13*W33)*(J22*W21 + J23*W31)*(a*c + b**2)/(a*c - b**2)**2) + 2*R12**2*s_y*(a*b*(J22*W21 + J23*W31)*(J22*W22 + J23*W32)/(a*c - b**2)**2 + b*c*(J11*W11 + J13*W31)*(J11*W12 + J13*W32)/(a*c - b**2)**2 - (J11*W11 + J13*W31)*(J22*W22 + J23*W32)*(a*c + b**2)/(a*c - b**2)**2 - (J11*W12 + J13*W32)*(J22*W21 + J23*W31)

#### (ii.3) $\frac{\partial \sigma_{\mu}}{\partial z}$ analogamente a (ii.1)

#### (iii.3) = (iii.1) $\frac{\partial \sigma_{\mu}}{\partial s}$

#### (iv.3) $\frac{\partial s}{\partial z}$ analogamente a (iv.1)

#### (iv.3.a) $\frac{\partial z}{\partial \Sigma'}$ analogamente a (iv.1.a)

#### (iv.3.b) = (iv.1.b) $\frac{\partial  \Sigma'}{\partial  \Sigma}$

#### (iv.3.c) = (iv.1.c) $\frac{\partial  \Sigma}{\partial s}$

#### (v.3) Resultado de $\frac{\partial z}{\partial s} $

In [57]:
dz_ds = dy_dCov2D * sp.Matrix(dCov2D_dCov3D) * sp.Matrix(dCov3D_ds)
# print(sp.latex(dz_ds))
print()
dz_ds.T

Matrix([
[2*R11**2*s_x*(a*b*(J22*W21 + J23*W31)**2/(a*c - b**2)**2 + b*c*(J11*W11 + J13*W31)**2/(a*c - b**2)**2 - 2*(J11*W11 + J13*W31)*(J22*W21 + J23*W31)*(a*c + b**2)/(a*c - b**2)**2) + 2*R11*R21*s_x*(a*b*(J22*W21 + J23*W31)*(J22*W22 + J23*W32)/(a*c - b**2)**2 + b*c*(J11*W11 + J13*W31)*(J11*W12 + J13*W32)/(a*c - b**2)**2 - (J11*W11 + J13*W31)*(J22*W22 + J23*W32)*(a*c + b**2)/(a*c - b**2)**2 - (J11*W12 + J13*W32)*(J22*W21 + J23*W31)*(a*c + b**2)/(a*c - b**2)**2) + 2*R11*R31*s_x*(a*b*(J22*W21 + J23*W31)*(J22*W23 + J23*W33)/(a*c - b**2)**2 + b*c*(J11*W11 + J13*W31)*(J11*W13 + J13*W33)/(a*c - b**2)**2 - (J11*W11 + J13*W31)*(J22*W23 + J23*W33)*(a*c + b**2)/(a*c - b**2)**2 - (J11*W13 + J13*W33)*(J22*W21 + J23*W31)*(a*c + b**2)/(a*c - b**2)**2) + 2*R12**2*s_y*(a*b*(J22*W21 + J23*W31)*(J22*W22 + J23*W32)/(a*c - b**2)**2 + b*c*(J11*W11 + J13*W31)*(J11*W12 + J13*W32)/(a*c - b**2)**2 - (J11*W11 + J13*W31)*(J22*W22 + J23*W32)*(a*c + b**2)/(a*c - b**2)**2 - (J11*W12 + J13*W32)*(J22*W21 + J23*W31)